# Data quality

Run `.\.tools\uv.exe run python -m kalix.prepare` from the project root before opening this notebook.
Inspect the common-range working copies and quality report before choosing model-specific rules.
Every calendar day is retained; no values are interpolated or filtered by quality flags.

Both Nikkaluokta (`temperature_c`) and Tarfala (`tarfala_temperature_c`) are included.


In [ ]:
from kalix.data import load_processed_data, project_root

ROOT = project_root()
daily = load_processed_data(ROOT)
measurements = [
    "temperature_c",
    "tarfala_temperature_c",
    "precipitation_mm",
    "discharge_m3_s",
]

In [ ]:
daily.head()

In [ ]:
summary = daily[measurements].describe().T
summary["missing_days"] = daily[measurements].isna().sum()
summary["missing_percent"] = daily[measurements].isna().mean() * 100
summary

In [ ]:
quality_columns = [name for name in daily.columns if name.endswith("_quality")]
for name in quality_columns:
    print(name)
    print(daily[name].value_counts(dropna=False))
    print()

In [ ]:
missing_columns = [name for name in daily.columns if name.endswith("_missing")]
daily.loc[daily[missing_columns].any(axis=1), measurements + missing_columns]

In [ ]:
interval_columns = [name for name in daily.columns if name.endswith("_utc")]
daily[interval_columns].head()

## Before modeling

Review gaps by season and source quality definitions. Daily labels describe different measurement windows; inspect the UTC intervals and discharge date convention before defining forecast inputs.

`daily.attrs["sources"]` contains source metadata. All temperature values currently have flag `Y`; a blanket `G` filter would discard the series. Recent discharge flagged `O` remains separate from missing values.

Keep missing calendar rows and real zero precipitation. Later, require complete input and target windows for each training sample instead of discarding whole years automatically. Incomplete precipitation totals must remain unavailable. Missing winter weather can affect calculated snow storage beyond the gap itself.

No filling is performed here. A later experiment could test bounded temperature gaps of at most three days in separate estimated-value columns; respect forecast-time availability and training/validation boundaries. June 1999 precipitation remains missing.


In [ ]:
from IPython.display import Markdown, display

report_path = ROOT / "data" / "processed" / "quality_report.md"
display(Markdown(report_path.read_text(encoding="utf-8")))